# RDT-1B attribution study: run everything in one cell
Runs setup, every study stage and the report. **Re-run the cell after a disconnect**: results are restored from
Google Drive and finished work is skipped, so it continues where it stopped.

Removed patches are **inpainted** from the rest of the frame (method-neutral) instead of filled with grey squares;
stage B re-scores the saved maps with both fills. Stage D hides 5% of patches with magnitude-ranked conditions;
stage F adds the modality share at every decision of whole episodes. See `docs/ITERATIONS.md` for what each run
showed.

Pilot 3 (resuming pilot 2 with these changes, `D_EPISODES = 10`) took roughly 2.5 h on an A100.

In [ ]:
#@title Run everything (setup, stages, report). Re-run after a disconnect to resume.
OUT_DRIVE = '/content/drive/MyDrive/rdt-xai/pilot2'  #@param {type:"string"}
ENV = 'rdt_era'           #@param ["rdt_era", "current"]
STAGES = 'verify,A,B,C,D,E,F'  #@param {type:"string"}
TASKS = ['PickCube-v1', 'StackCube-v1', 'PushCube-v1']  #@param
N_EPISODES = 16           #@param {type:"integer"}
B_MOMENTS = 60            #@param {type:"integer"}
B_STABILITY_MOMENTS = 24  #@param {type:"integer"}
N_EVALS = 16              #@param {type:"integer"}
FAST = True               #@param {type:"boolean"}
C_MOMENTS = 30            #@param {type:"integer"}
D_TASKS = ['PickCube-v1', 'StackCube-v1']  #@param
D_EPISODES = 16           #@param {type:"integer"}
D_FRAC = 0.05             #@param {type:"number"}
E_MOMENTS = 8             #@param {type:"integer"}
F_MOMENTS = 30            #@param {type:"integer"}
F_TIMELINE_EPISODES = 8   #@param {type:"integer"}

import os, sys, json, subprocess, importlib
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/rdt-igtesting'
if not os.path.exists(REPO):
    subprocess.run(f'git clone -q -b cvpr-xai https://github.com/rdgbrandon/rdt-igtesting {REPO}', shell=True, check=True)
subprocess.run(f'cd {REPO} && git fetch -q origin cvpr-xai && git checkout -q cvpr-xai && git pull -q origin cvpr-xai',
               shell=True, check=True)
sys.path.insert(0, REPO)
import rdt_xai.colab as colab
importlib.reload(colab)

OUT = colab.setup(OUT_DRIVE)                 # installs, restores earlier results from Drive
colab.download_models()
PY = colab.rdt_era_env() if ENV == 'rdt_era' else sys.executable
CFG = f'{OUT}/config.json'
json.dump(dict(out=OUT, env=ENV, tasks=TASKS, n_episodes=N_EPISODES, b_moments=B_MOMENTS,
               b_stability_moments=B_STABILITY_MOMENTS, n_evals=N_EVALS, fast=FAST, c_moments=C_MOMENTS,
               d_tasks=D_TASKS, d_episodes=D_EPISODES, d_frac=D_FRAC, e_moments=E_MOMENTS, f_moments=F_MOMENTS,
               f_timeline_episodes=F_TIMELINE_EPISODES), open(CFG, 'w'), indent=1)
for stage in [s.strip() for s in STAGES.split(',') if s.strip()]:
    print(f'\n===== stage {stage} =====', flush=True)
    colab.run_stage(PY, CFG, stage, OUT_DRIVE)
    if stage == 'verify':
        from IPython.display import Image, display
        display(Image(f'{OUT}/verify_render.png', width=256))

import rdt_xai.report
importlib.reload(rdt_xai.report)             # pick up report fixes pulled above
colab.show_report(OUT, OUT_DRIVE)


In [ ]:
#@title (Optional) Rebuild only the report from Drive - no GPU, installs or models needed (~1-2 min)
OUT_DRIVE = '/content/drive/MyDrive/rdt-xai/pilot2'  #@param {type:"string"}
import os, sys, subprocess, importlib
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/rdt-igtesting'
if not os.path.exists(REPO):
    subprocess.run(f'git clone -q -b cvpr-xai https://github.com/rdgbrandon/rdt-igtesting {REPO}', shell=True, check=True)
subprocess.run(f'cd {REPO} && git pull -q origin cvpr-xai', shell=True, check=True)
sys.path.insert(0, REPO)
import rdt_xai.colab as colab, rdt_xai.report
importlib.reload(colab); importlib.reload(rdt_xai.report)
colab.show_report(OUT_DRIVE)                 # reads results from Drive and writes report/ there
